In [1]:
import pandas as pd
import numpy as np

house = pd.read_csv("kc_house_data.csv")   # file notebook ke folder mein rakhein

print(house.shape)
print(house.columns.tolist())
print(house.dtypes)
print(house.isnull().sum().sum())
house.head(3)

(21613, 20)
['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']
date                 str
price              int64
bedrooms           int64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
grade              int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
zipcode            int64
lat              float64
long             float64
sqft_living15      int64
sqft_lot15         int64
dtype: object
0


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,20141013T000000,7700000,6,8.00,12050,27600,2.5,0,3,4,13,8570,3480,1910,1987,98102,47.6298,-122.323,3940,8800
1,20140611T000000,7062500,5,4.50,10040,37325,2.0,1,2,3,11,7680,2360,1940,2001,98004,47.6500,-122.214,3930,25449
2,20140919T000000,6885000,6,7.75,9890,31374,2.0,0,4,3,13,8860,1030,2001,0,98039,47.6305,-122.240,4540,42730


In [2]:
if "date" in house.columns:
    # 1. Date text se asal date banayein (KC House mein format 20141013T000000 hota hai)
    house["sale_date"] = pd.to_datetime(house["date"].str[:8], format="%Y%m%d", errors="coerce")
    print("Date parse fail:", house["sale_date"].isna().sum())
    assert house["sale_date"].notna().all(), "Date format match nahi hua, house['date'].head() dekhein"

    # 2. Naye features
    house["house_age"] = house["sale_date"].dt.year - house["yr_built"]
    house["was_renovated"] = (house["yr_renovated"] > 0).astype(int)

    # 3. Ab purane columns hatayein
    house = house.drop(columns=["date", "sale_date", "yr_built", "yr_renovated", "zipcode"])
else:
    print("Yeh cell pehle chal chuka hai.")

print(house.shape)
print(house["house_age"].describe())
print("Price skew:", house["price"].skew())
print("Log price skew:", np.log1p(house["price"]).skew())

Date parse fail: 0
(21613, 18)
count    21613.000000
mean        43.317818
std         29.375493
min         -1.000000
25%         18.000000
50%         40.000000
75%         63.000000
max        115.000000
Name: house_age, dtype: float64
Price skew: 4.024069144684711
Log price skew: 0.4280765149574702


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

house["house_age"] = house["house_age"].clip(lower=0)

X = house.drop(columns=["price"])
y = np.log1p(house["price"])          # target log mein

feature_cols = list(X.columns)        # tarteeb save kar rahe hain (dashboard ke liye)
print(len(feature_cols), feature_cols)

X_train, X_other, y_train, y_other = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_other, y_other, test_size=0.5, random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)

17 ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'lat', 'long', 'sqft_living15', 'sqft_lot15', 'house_age', 'was_renovated']
(17290, 17) (2161, 17) (2162, 17)


In [4]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

inputs  = keras.Input(shape=(X_train.shape[1],), name="input_layer")
hidden1 = layers.Dense(64, activation="relu", name="hidden_layer1")(inputs)
hidden2 = layers.Dense(32, activation="relu", name="hidden_layer2")(hidden1)
concat  = layers.Concatenate()([inputs, hidden2])               # skip connection
output  = layers.Dense(1, name="price_output")(concat)          # activation nahi (linear)

reg_model = keras.Model(inputs=inputs, outputs=output)
reg_model.summary()

reg_model.compile(
    loss="mse",
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["mae"]
)

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
chk_pt = keras.callbacks.ModelCheckpoint("house_price.keras", save_best_only=True, monitor="val_loss")

history = reg_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=64,
    callbacks=[early_stop, chk_pt]
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 17)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hidden_layer1       │ (None, 64)        │      1,152 │ input_layer[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hidden_layer2       │ (None, 32)        │      2,080 │ hidden_layer1[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 49)        │          0 │ input_layer[0][0… │
│ (Concatenate)       │                   │            │ hidden_layer2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price_output        │ (None, 1)         │         50 │ concatenate[0][0] │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,282 (12.82 KB)

 Trainable params: 3,282 (12.82 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
271/271 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 32.3064 - mae: 4.1554 - val_loss: 3.7553 - val_mae: 1.5087
Epoch 2/200
271/271 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 2.9724 - mae: 1.2603 - val_loss: 1.9284 - val_mae: 1.0561
Epoch 3/200
271/271 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 1.6810 - mae: 0.9357 - val_loss: 1.1619 - val_mae: 0.8072
Epoch 4/200
271/271 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.9964 - mae: 0.7118 - val_loss: 0.6962 - val_mae: 0.6106
Epoch 5/200
271/271 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.5857 - mae: 0.5378 - val_loss: 0.4150 - val_mae: 0.4595
Epoch 6/200
271/271 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.3438 - mae: 0.4077 - val_loss: 0.2542 - val_mae: 0.3528
Epoch 7/200
271/271 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.2136 - mae: 0.3212 - val_loss: 0.1669 - val_mae: 0.2882
Epoch 8/200
271/271 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1463 - mae: 0.2699 - val_loss: 0.1195 - val_mae: 0.2481
Epoch 9/200
271/271 ━━━━━━━━━━━━━━━━━━━━ 1s 5m

In [6]:
from sklearn.metrics import mean_absolute_error, r2_score

best_model = keras.models.load_model("house_price.keras")

pred   = np.expm1(best_model.predict(X_test).ravel())   # log se wapas dollars
actual = np.expm1(y_test)

print(f"Test MAE: ${mean_absolute_error(actual, pred):,.0f}")
print(f"Test R2 : {r2_score(actual, pred):.4f}")
print(f"Median % error: {np.median(np.abs(pred - actual) / actual) * 100:.1f}%")

68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
Test MAE: $74,994
Test R2 : 0.8395
Median % error: 10.0%


In [7]:
import joblib, json

joblib.dump(scaler, "house_scaler.pkl")

meta = {
    "columns": feature_cols,
    "ranges": {c: [float(X[c].min()), float(X[c].max()), float(X[c].median())]
               for c in feature_cols}
}
with open("house_meta.json", "w") as f:
    json.dump(meta, f)

print(feature_cols)

['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'lat', 'long', 'sqft_living15', 'sqft_lot15', 'house_age', 'was_renovated']


In [9]:
import sys
print(sys.executable)


c:\ProgramData\anaconda3\python.exe
